# 05 — Calibration

Are the model's percentages honest? If it says 70% for a hundred fights, fighter A
should win about seventy. Calibration learns one number, the **stretch** `a`, and
redraws the curve that turns the model's score into a percentage:

    p_new = 1 / (1 + e^-(a × score))      a > 1 bolder · a < 1 more cautious · a = 1 no change

No shift (that would favour corner A), no retraining, and no pick can change, so
accuracy and AUC stay put. Only log loss can move. Code: `modeling/calibrate.py`.

The validation table in `04_elo.ipynb` looked strongly under-confident (said 64%,
won 74.5%). Testing inside the training years before building this told a calmer
story: a ≈ 1.12, with a 90% range that includes 1.0. So `a` is fitted on the
**training years only** and the validation years get one look.

**Decision rule, set before running:** keep the calibrator if validation log loss on
men's fights improves. Otherwise save `a = 1.0` (no change) and record why.

In [1]:
import numpy as np
import pandas as pd

from ufc_predictor.data.io import load_processed
from ufc_predictor.features import manifest
from ufc_predictor.modeling.calibrate import fit_stretch, save_calibration, stretch, stretch_interval
from ufc_predictor.modeling.evaluate import breakdown, calibration_table, has_debutant, is_womens_fight, scorecard
from ufc_predictor.modeling.model import fit_models, predict_proba_mean
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import load_params, out_of_fold_predictions

featured = load_processed("fights_featured")
train, validation, _ = chronological_split(featured)      # the test years stay closed
params = load_params()
print(f"training fights: {len(train):,} | validation fights: {len(validation):,}")

training fights: 4,309 | validation fights: 1,017


## 1. Out-of-fold predictions

The calibrator must learn from predictions made on fights the model never saw. The
same expanding-window folds as tuning (train 2014–18 → predict 2019, …, train
2014–21 → predict 2022), each prediction averaged over 8 seeds, exactly like the
final model's.

In [2]:
y_oof, p_oof, year = out_of_fold_predictions(train, params)
print(f"{len(y_oof):,} out-of-fold predictions, years {year.min()}-{year.max()}")

1,953 out-of-fold predictions, years 2019-2022


## 2. Fit the stretch, with its uncertainty

The 90% range is how far chance alone moves `a` (resample the fights, refit, repeat).
The per-year values show how much one year on its own can wander, which is why one
`a` is used for every year.

In [3]:
a = fit_stretch(y_oof, p_oof)
interval = stretch_interval(y_oof, p_oof)
print(f"a = {a:.3f}   90% range {interval[0]:.2f} - {interval[1]:.2f}")

by_year = pd.DataFrame([
    {"year": int(yr), "fights": int((year == yr).sum()), "a_that_year_alone": round(fit_stretch(y_oof[year == yr], p_oof[year == yr]), 2)}
    for yr in np.unique(year)
])
by_year

a = 1.077   90% range 0.95 - 1.25


,year,fights,a_that_year_alone
0,2019,506,0.85
1,2020,444,1.03
2,2021,497,1.20
3,2022,506,1.26


In [4]:
# out-of-fold calibration, before and after the stretch
pd.concat({"before": calibration_table(y_oof, p_oof), "after": calibration_table(y_oof, stretch(p_oof, a))}, axis=1)

before                   after                 
  fights predicted actual fights predicted actual
0    391     0.320  0.281    391     0.308  0.281
1    390     0.427  0.408    390     0.421  0.408
2    391     0.501  0.499    391     0.501  0.499
3    390     0.573  0.559    390     0.578  0.559
4    391     0.678  0.680    391     0.690  0.680

## 3. One look at validation

The final predictor: trained on 2014–2022, 8 seeds, each asked both ways round.
Raw versus stretched, men's fights first.

In [5]:
X_train, y_train = manifest.make_xy(train)
X_val, y_val = manifest.make_xy(validation)
mens = ~is_womens_fight(validation, history=featured).to_numpy()

p_raw = predict_proba_mean(fit_models(X_train, y_train, params), X_val)
p_cal = stretch(p_raw, a)
assert ((p_raw > 0.5) == (p_cal > 0.5)).all(), "a stretch must never change a pick"

card = scorecard({"raw": p_raw, "calibrated": p_cal}, y_val, mens)
card

,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
raw,0.6464,0.6402,0.6892,0.6372,0.6450,0.6778
calibrated,0.6464,0.6389,0.6892,0.6372,0.6441,0.6778


Every evaluation also splits the fights by whether a debutant is involved: a fighter with
no UFC history, for whom 30 of the 40 inputs are blank. Men's fights.

In [6]:
breakdown({"raw": p_raw, "calibrated": p_cal}, y_val, has_debutant(validation), keep=mens)

fights  accuracy  log_loss     auc
group           predictions                                    
all             raw             837    0.6464    0.6402  0.6892
                calibrated      837    0.6464    0.6389  0.6892
no debutant     raw             670    0.6537    0.6386  0.6913
                calibrated      670    0.6537    0.6372  0.6913
with a debutant raw             167    0.6168    0.6467  0.6825
                calibrated      167    0.6168    0.6455  0.6825

In [7]:
y = np.asarray(y_val, dtype=bool)
pd.concat({"before": calibration_table(y[mens], p_raw[mens]), "after": calibration_table(y[mens], p_cal[mens])}, axis=1)

before                   after                 
  fights predicted actual fights predicted actual
0    168     0.318  0.244    168     0.306  0.244
1    167     0.429  0.359    167     0.424  0.359
2    167     0.500  0.461    167     0.500  0.461
3    167     0.570  0.551    167     0.575  0.551
4    168     0.682  0.696    168     0.695  0.696

## 4. The decision

Applied exactly as written at the top.

In [8]:
gain = card.loc["raw", "men_log_loss"] - card.loc["calibrated", "men_log_loss"]
if gain > 0:
    decision = f"kept: men's validation log loss improved by {gain:.4f}"
    save_calibration(a, interval, len(y_oof), decision)
else:
    decision = f"not kept: men's validation log loss changed by {-gain:+.4f}; the model stays uncalibrated (a = 1.0)"
    save_calibration(1.0, interval, len(y_oof), decision + f" (fitted a was {a:.3f})")
print(decision)

kept: men's validation log loss improved by 0.0013
